In [4]:
"""
HF_TOKEN 검증 노트북

목적: 쉘 환경 변수로 등록한 HF_TOKEN을
      Python(Jupyter) 프로세스에서 제대로 읽어올 수 있는지 확인.
"""

import os

# os.environ: 현재 프로세스가 상속받은 환경 변수 딕셔너리
token = os.environ.get("HF_TOKEN")

if token is None:
    print("FAIL: HF_TOKEN 환경 변수가 Python 프로세스에 안 보임")
    print("→ VS Code를 완전 종료 후 재시작해보세요 (.bashrc 재로드 필요)")
elif not token.startswith("hf_"):
    print(f"FAIL: 토큰 형식 이상함. 앞 3글자: {token[:3]}")
else:
    print(f"OK: HF_TOKEN 인식됨")
    print(f"  - 길이: {len(token)} 자")
    print(f"  - 앞 3글자: {token[:3]}...")
    print(f"  - 출처: os.environ (코드에 평문 없음)")

OK: HF_TOKEN 인식됨
  - 길이: 37 자
  - 앞 3글자: hf_...
  - 출처: os.environ (코드에 평문 없음)


In [ ]:
"""
[3-1] 토크나이저 다운로드 + 캐시 경로 검증

목적: 본체 9GB 받기 전에 5MB 토크나이저로 HF 인증/네트워크/캐시를 검증.
     성공하면 본체 다운로드도 안전.
"""

"""
[ 토크나이저 ]
- 사람의 글을 모델이 먹을 수 있는 숫자로 바꿔주는 번역기
- 모델: 글 못 읽음 => 숫자만 처리 가능 

2가지 기능)
1. 글 -> 토큰(의미 단위) -> 숫자(토큰 ID)
2. 숫자(토큰 ID) -> 토큰(의미 단위) -> 글
"""

import os
from pathlib import Path
from transformers import AutoTokenizer

# 모델 ID는 변수로 분리 (이후 셀에서도 재사용 + 변종 바꾸기 쉬움)
MODEL_ID = "google/gemma-4-E4B-it"

# HF 캐시 위치 확인 (기본: ~/.cache/huggingface/)
hf_cache_root = Path(os.environ.get("HF_HOME", "~/.cache/huggingface")).expanduser()
print(f"HF 캐시 루트: {hf_cache_root}")
print(f"  - 존재 여부: {hf_cache_root.exists()}")
print()

# 토크나이저 다운로드 시도
print(f"토크나이저 다운로드 시작: {MODEL_ID}")
print("(처음이면 5~10초 소요, 캐시되어 있으면 즉시)")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=os.environ["HF_TOKEN"],  # 환경변수에서 토큰 가져옴
)

# 검증: 토크나이저 핵심 정보 출력
print()
print("토크나이저 로드 성공")
print(f"  - 클래스: {type(tokenizer).__name__}")
print(f"  - vocab 크기: {tokenizer.vocab_size:,} 토큰")
print(f"  - 모델 최대 입력: {tokenizer.model_max_length:,} 토큰")
print(f"  - 특수 토큰 (BOS/EOS): {tokenizer.bos_token!r} / {tokenizer.eos_token!r}")

HF 캐시 루트: /home/xorms/.cache/huggingface
  - 존재 여부: True

토크나이저 다운로드 시작: google/gemma-4-E4B-it
(처음이면 5~10초 소요, 캐시되어 있으면 즉시)

토크나이저 로드 성공
  - 클래스: GemmaTokenizer
  - vocab 크기: 262,144 토큰
  - 모델 최대 입력: 1,000,000,000,000,000,019,884,624,838,656 토큰
  - 특수 토큰 (BOS/EOS): '<bos>' / '<eos>'


In [6]:
import bitsandbytes as bnb
print(f"bitsandbytes: {bnb.__version__}")
print("OK - 4-bit 양자화 사용 가능")

bitsandbytes: 0.49.2
OK - 4-bit 양자화 사용 가능


In [ ]:
"""
[3-2 재시도] Gemma 4 E4B-it 4-bit 양자화 로드

배경: BF16(14.79 GB) → 12 GB GPU에 swap 발생 → 추론 시 OOM
해결: NF4 4-bit 양자화 → 약 4~5 GB → 여유롭게 적재

NF4 (Normal Float 4)란?
- 4-bit 정수가 아니라 4-bit "정규분포 매핑" 양자화
- LLM 가중치 분포(거의 정규분포)에 최적화
- QLoRA 논문에서 검증된 방식. 성능 손실 최소
"""

import os
import time
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E4B-it"

# 양자화 설정
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                          # 4-bit 양자화 활성화 -> 가중치를 4-bit로 저장
    bnb_4bit_quant_type="nf4",                  # NF4 (Normal Float 4 - LLM 가중치 분포에 최적화된 양자화 방식) 사용
    bnb_4bit_compute_dtype=torch.bfloat16,      # 실제 계산은 BF16으로 (정확도 보존) -> 저장은 4bit, 계산은 BF16
                                                # -> 이렇게 하면 계산 정확도는 유지 + 메모리 줄임 -> QLoRA 논문의 핵심 아이디어
    bnb_4bit_use_double_quant=True,             # 양자화 상수도 양자화 (추가 메모리 절약)
)

print(f"모델 로드 시작: {MODEL_ID}")
print(f"  - 양자화: NF4 4-bit")
print(f"  - compute dtype: bfloat16")
print(f"  - 예상 VRAM: 4~5 GB")
print()

start = time.time()

model = AutoModelForCausalLM.from_pretrained( # AutoModelForCausalLM: "다음 토큰 예측" 모델 클래스(GPT/Llama/Gemma 다 이거)
    MODEL_ID,
    token=os.environ["HF_TOKEN"],
    quantization_config=quant_config,
    device_map="cuda", # 모델 전체를 GPU에 올림
)

elapsed = time.time() - start

# 검증
print(f"\n모델 로드 완료 ({elapsed:.1f}초)")
print(f"  - 클래스: {type(model).__name__}")

# VRAM 점유 확인
allocated_gb = torch.cuda.memory_allocated() / (1024**3)
reserved_gb = torch.cuda.memory_reserved() / (1024**3)
print(f"  - GPU 점유: {allocated_gb:.2f} GB (allocated) / {reserved_gb:.2f} GB (reserved)")
print(f"  - 남은 VRAM: {11.94 - reserved_gb:.2f} GB (추론 여유)")

모델 로드 시작: google/gemma-4-E4B-it
  - 양자화: NF4 4-bit
  - compute dtype: bfloat16
  - 예상 VRAM: 4~5 GB



Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

/home/xorms/miniconda3/envs/ai/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



모델 로드 완료 (9.4초)
  - 클래스: Gemma4ForConditionalGeneration
  - GPU 점유: 8.67 GB (allocated) / 8.79 GB (reserved)
  - 남은 VRAM: 3.15 GB (추론 여유)


In [ ]:
"""
[3-3] 한국어 hello world + 추론 속도 측정 (수정판)

수정 사유: transformers 5.x + 멀티모달 토크나이저는 BatchEncoding 반환.
         .shape 직접 접근 대신 ["input_ids"].shape 사용.
"""

import torch
import time

# 1. 입력 준비(ENCODE (글 -> 숫자))
messages = [
    {"role": "user", "content": "안녕하세요! 짧게 자기소개 부탁드려요."}
]

inputs = tokenizer.apply_chat_template( # apply_chat_template: messages를 Gemma4 전용 형식으로 변환
    messages,
    add_generation_prompt=True,
    return_tensors="pt",             # pytorch tensor로 변환
    return_dict=True,                # dict(BatchEncoding) 형식으로 명시
).to(model.device)                   # tensor를 gpu로 변환

# inputs는 dict 형태: {"input_ids": ..., "attention_mask": ...}
input_length = inputs["input_ids"].shape[1]
print(f"입력 토큰 수: {input_length}")
print(f"입력 키들: {list(inputs.keys())}")  # 어떤 키 있는지 확인용
print()

# 2. 생성
print("생성 중...")
start = time.time()

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,                     # dict 통째로 unpack해서 전달
        max_new_tokens=80,
        do_sample=False, # greedy decoding: 확률 가장 높은 토큰 매번 선택 -> 재현성(같은 입력에 항상 같은 출력
        # do_sample = True: 확률 분포에서 샘플링 -> 매번 다른 답
    )

elapsed = time.time() - start

# 3. 디코딩
# output_ids는 (batch, seq_len) 텐서. 입력 + 출력이 다 있음
generated_ids = output_ids[0][input_length:]
generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True) # tokenizer.decode: 숫자 -> 토큰 -> 글자

num_new_tokens = len(generated_ids)
tokens_per_sec = num_new_tokens / elapsed

print(f"생성 완료 ({elapsed:.2f}초)")
print(f"  - 생성된 토큰 수: {num_new_tokens}")
print(f"  - 속도: {tokens_per_sec:.1f} tokens/sec")
print()
print("=" * 60)
print("Gemma 4 응답:")
print("=" * 60)
print(generated_text)
print("=" * 60)

# 4. VRAM 확인
reserved_gb = torch.cuda.memory_reserved() / (1024**3)
print(f"\n추론 후 VRAM: {reserved_gb:.2f} GB / 11.94 GB")

입력 토큰 수: 21
입력 키들: ['input_ids', 'attention_mask']

생성 중...


/home/xorms/miniconda3/envs/ai/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


생성 완료 (6.16초)
  - 생성된 토큰 수: 80
  - 속도: 13.0 tokens/sec

Gemma 4 응답:
안녕하세요! 저는 Google에서 훈련한 대규모 언어 모델, **Gemini**라고 합니다. 😊

저는 여러분의 질문에 답하고, 글을 쓰고, 정보를 요약하거나, 재미있는 대화를 나누는 등 다양한 방식으로 도움을 드릴 수 있어요.

**궁금한 것이 있으시거나 도움이 필요하시면 언제든지 말씀해주세요!** 어떤 이야기를 나누고 싶으신

추론 후 VRAM: 8.79 GB / 11.94 GB
